# Author: José Ángel de Bustos Pérez
# License: GNU General Public License v3.0

# Basic implementation for CKKS algorithm using Pyphel (https://pyfhel.readthedocs.io/)

CKKS (Cheon–Kim–Kim–Song, 2017) is a homomorphic encryption scheme based on RLWE (Ring Learning With Errors), like BFV, but designed to work with real or complex numbers in an approximate way. That means that the operations will add some error.

Key differences compared to BFV:

* Data: float/double, not integers.
* Results: approximate (there is rounding error).
* Rescaling: after each multiplication, rescaling is required.
* Levels: each rescale consumes a level of the ciphertext.
* Slots: n/2 instead of n (slots are complex conjugates).

It is the standard scheme for machine learning on encrypted data, statistics, signal processing, etc.

The first step is to set the parameters and generate the keys:

* **n**, polynomial degree (power of 2). In CKKS, the number of slots is n/2.
* **scale**, scaling factor. CKKS encodes floats by multiplying them by this factor to convert them into large integers. A larger scale gives more decimal precision but adds more noise. Typical scale value used: 2^30.
* **qi_sizes**, sizes (in bits) of the prime moduli that form the modulus chain. CKKS uses modulus switching: each rescale after a multiplication removes one prime from the chain. The number of intermediate primes equals the multiplicative depth you can support.
* **sec**, security level in bits.

In [1]:
from Pyfhel import Pyfhel
import numpy as np

HE = Pyfhel()

ckks_params = {
    'scheme': 'CKKS',
    'n': 2**14,                # 16384 coefficients => 8192 slots for batching
    'scale': 2**30,            # approximate precision: ~9 decimal places
    # qi_sizes gives a chain of 7 primes. The first and last are “large” (60 bits) for technical reasons; 
    # the ones in the middle are the size of the scale (30 bits). ⇒ Approximately 5 levels of multiplication.
    'qi_sizes': [60, 30, 30, 30, 30, 30, 60],
    'sec': 128,
}

HE.contextGen(**ckks_params)
HE.keyGen()           # Key creation
HE.rotateKeyGen()     # Required for rotations (we will use it in batching)
HE.relinKeyGen()      # Required to reduce size after multiplication

print(f"Scheme: {HE.scheme}")
print(f"n (Polynomial degree): {HE.n}")
print(f"Available Slots (n/2): {HE.n // 2}")
print(f"Scale: 2^{int(np.log2(HE.scale))}")
print(f"qi modulus chain: {ckks_params['qi_sizes']}")
print(f"=> Available multiplicative depth: " f"{len(ckks_params['qi_sizes']) - 2}")

Scheme: Scheme_t.ckks
n (Polynomial degree): 16384
Available Slots (n/2): 8192
Scale: 2^30
qi modulus chain: [60, 30, 30, 30, 30, 30, 60]
=> Available multiplicative depth: 5


Unlike BFV, in CKKS the **noise budget** is not measured in the same way. What matters is the LEVEL of the ciphertext (how many primes remain in the modulus chain). Each rescale consumes one level. When you reach the last level, you can no longer perform multiplications.

## Basic operations

We will perform basic operations such as addition and multiplications to check how noise is increasing. We will start with addition:

In [2]:
import random

# Randon number generation
a = random.random()
b = random.random()

expected_value = a + b

# Encrypting data. CKKS always works with vectors. To encrypt a scalar, we place it
# in an array (the remaining slots are filled with zeros).
fhe_a = HE.encryptFrac(np.array([a], dtype=np.float64))
fhe_b = HE.encryptFrac(np.array([b], dtype=np.float64))

# Homomorphic operation (addition)
fhe_suma = fhe_a + fhe_b

# Getting the operation value, decrypting
suma = HE.decryptFrac(fhe_suma)[0]

# Error
error = abs(expected_value - suma)

print(f"Plaintext data: a = {a}, b = {b}")
print(f"Expected value: {expected_value}")
print(f"Homomorphic addition: {suma}")
print(f"Error: {error:.2e}")

Plaintext data: a = 0.9943479663279963, b = 0.9024574390292777
Expected value: 1.896805405357274
Homomorphic addition: 1.8968000793530626
Error: 5.33e-06


We are going to proceed with multiplication.

In [3]:
expected_value = a * b

# Homomorphic multiplication
fhe_mult = fhe_a * fhe_b
# Realinarization
~fhe_mult

# Rescalation. After each multiplication, we must manually rescale. CKKS multiplies two numbers scaled by 
# S, and the result ends up scaled by S^2. The rescale operation divides by S and removes one prime from the 
# chain to keep the original scaling factor S
HE.rescale_to_next(fhe_mult)

# Getting the operation value, decrypting
mult = HE.decryptFrac(fhe_mult)[0]

# Error
error = abs(expected_value - mult)

print(f"Plaintext data: a = {a}, b = {b}")
print(f"Expected value: {expected_value}")
print(f"Homomorphic multiplication: {mult}")
print(f"Error: {error:.2e}")

Plaintext data: a = 0.9943479663279963, b = 0.9024574390292777
Expected value: 0.897356719196334
Homomorphic multiplication: 0.8973539781871963
Error: 2.74e-06


## Performing homomorphic encryption with multiple operations

We are going to show how to perform more complex operations with homomorphic encryption. Let's assume we want to perform homomorphic encryption to:

$$f(x,y) = (x+y)^2 - (x-y)^2$$

In [4]:
# Random number generation
x = random.random()
y = random.random()

expected_value = (x + y)**2 - (x-y)**2

# Encrypting data
fhe_x = HE.encryptFrac(np.array([x], dtype=np.float64))
fhe_y = HE.encryptFrac(np.array([y], dtype=np.float64))

# (x + y)
fhe_addition_xy = fhe_x + fhe_y

# (x + y)^2
fhe_square_add = fhe_addition_xy * fhe_addition_xy
# Realinarization
~fhe_square_add
# Rescaling, uses one prime in the modulus chain
HE.rescale_to_next(fhe_square_add)

# (x - y)
fhe_substraction_xy = fhe_x - fhe_y

# (x - y)^2
fhe_square_substraction = fhe_substraction_xy * fhe_substraction_xy
# Realinarization
~fhe_square_substraction
# Rescaling, uses one prime in the modulus chain
HE.rescale_to_next(fhe_square_substraction)

# Final. Before doing the substraction both ciphertext must have the same
# level, in both we have performed one rescale operation so they are at the
# same level.
fhe_final = fhe_square_add - fhe_square_substraction

# Getting the operation value, decrypting
final = HE.decryptFrac(fhe_final)[0]

# Error
error = abs(expected_value - final)

print(f"Operation: ({x} + {y})**2 - ({x}-{y})**2")
print(f"Expected value: {expected_value}")
print(f"Homomorphic value: {final}")
print(f"Error: {error:.2e}")

Operation: (0.6773250359131189 + 0.5943237858987185)**2 - (0.6773250359131189-0.5943237858987185)**2
Expected value: 1.610201518511481
Homomorphic value: 1.6101992237531872
Error: 2.29e-06


Added noise is mostly the same that the noise added by the homomorphic multiplication. Now, we will perform the following:

$$f(x,y) = (x+y)^2 - (x-y)^3$$

In this example $(x+y)^2$ has a multiplicativity depth of 1 but $(x-y)^3$ has a multiplicativity depth of 2. That means that we will need to align levels (working on the same scale) to operate with them.

In [5]:
expected_value_third = (x + y)**2 - (x - y)**3

# Aligning data on the same scale. fhe_substraction_xy is in level 1
# but fhe_square_substraction is in level 2
fhe_substraction_xy_aligned = fhe_substraction_xy.copy()
HE.mod_switch_to_next(fhe_substraction_xy_aligned)

# (x - y)^3
fhe_third_substraction = fhe_square_substraction * fhe_substraction_xy_aligned
# Realinarization
~fhe_third_substraction
# Rescaling, uses one prime in the modulus chain
HE.rescale_to_next(fhe_third_substraction)

# (x + y)^2 is in level 1 (one rescaling) but (x - y)^3 is in level 2
# before operating them, we need to have both of them on the  same level
fhe_square_add_aligned = fhe_square_add.copy()
HE.mod_switch_to_next(fhe_square_add_aligned)

# Final
fhe_final = fhe_square_add_aligned - fhe_third_substraction

# Getting the operation value, decrypting
final = HE.decryptFrac(fhe_final)[0]

# Error
error = abs(expected_value_third - final)

print(f"Operation: ({x} + {y})**2 - ({x}-{y})**3")
print(f"Expected value: {expected_value_third}")
print(f"Homomorphic value: {final}")
print(f"Error: {error:.2e}")

Operation: (0.6773250359131189 + 0.5943237858987185)**2 - (0.6773250359131189-0.5943237858987185)**3
Expected value: 1.6165189131809974
Homomorphic value: 1.6166634644393885
Error: 1.45e-04


## Batching operation

Batching allows us to perform the same homomorphic operation to several data at the same time, parallelism.

If we are familiar with processor architectures we will know what **SIMD** (**S**imple **I**nstruction **M**ultiple **D**ata) is. One single operation using only one clock cicle operate on multiple data. Batching is the same but used in homomorphic encryption and it is used to speed up operations.

In [6]:
# Randon number generation
size = 10 # number of floats to operate at the same time
v1 = np.random.uniform(-100,  100, (size))
v2 = np.random.uniform(-100,  100, (size))

# Encrypting data
fhe_v1 = HE.encryptFrac(v1)
fhe_v2 = HE.encryptFrac(v2)

# Item by item homomorphic addition
fhe_add_vectorial = fhe_v1 + fhe_v2

# Getting the operation value, decrypting
final = HE.decryptFrac(fhe_add_vectorial)[:size]

# Error
expected_value = v1 + v2
error = np.abs(final - expected_value)

print(f"v1: {v1}")
print(f"v2: {v2}")
print(f"Homomorphic v1 + v2:  {final}")
print(f"Expected value: {expected_value}")
print(f"Error: {error}")

v1: [ -9.103999   -93.47867102  87.92989464  84.22205723 -91.23483409
  32.29068894  55.75724773   7.52593578 -74.87742045  16.61025063]
v2: [ 41.48130845  -0.74489571  87.22547357 -75.17795092  30.85156581
  83.72927726  94.35592581 -28.78446052 -17.90394538 -64.99481826]
Homomorphic v1 + v2:  [ 32.37730784 -94.22356721 175.15536902   9.04411156 -60.38326722
 116.01996477 150.113175   -21.2585224  -92.78136559 -48.38456691]
Expected value: [ 32.37730945 -94.22356673 175.15536821   9.04410632 -60.38326829
 116.01996621 150.11317354 -21.25852474 -92.78136582 -48.38456763]
Error: [1.60511254e-06 4.81908657e-07 8.09760792e-07 5.24137221e-06
 1.06408874e-06 1.44023517e-06 1.45748913e-06 2.34082648e-06
 2.33090418e-07 7.14335478e-07]


Let's see what happens with batching multiplication:

In [7]:
# Item by item homomorphic multiplication
fhe_multiplication_vectorial = fhe_v1 * fhe_v2
# Realinarization
~fhe_multiplication_vectorial

# Getting the operation value, decrypting
final = HE.decryptFrac(fhe_multiplication_vectorial)[:size]

# Error
expected_value = v1 * v2
error = np.abs(final - expected_value)

print(f"\nHomomorphic v1 * v2  = {final}")
print(f"Expected value = {v1 * v2}")
print(f"Error: {error}")


Homomorphic v1 * v2  = [ -377.64588851    69.6320103   7669.72677116 -6331.64158151
 -2814.73730723  2703.67572545  5261.0268524   -216.62992904
  1340.60123156 -1079.5803187 ]
Expected value = [ -377.64579088    69.63186076  7669.72670058 -6331.64168509
 -2814.7374878   2703.67604755  5261.02673005  -216.63000137
  1340.60124572 -1079.58022064]
Error: [9.76332187e-05 1.49531799e-04 7.05797202e-05 1.03576311e-04
 1.80577169e-04 3.22096169e-04 1.22343317e-04 7.23323438e-05
 1.41559344e-05 9.80594484e-05]


## Dot product or scalar product

Dot product or scalar product is a fundamental operation used in multiple algorithms such as linear regresion, support vector machines (SVM) or neural network algorithms. For this reason been able to operate it using homomorphic encription will ease using such algoritms with homomorphic encryption.

In [8]:
# Random sample
size = 10 # random sample size
v1 = np.random.uniform(-100,  100, (size))
v2 = np.random.uniform(-100,  100, (size))

# Encrypt data
fhe_v1 = HE.encryptFrac(v1)
fhe_v2 = HE.encryptFrac(v2)

# Cdot operation with encrypted data
fhe_cdot = fhe_v1 * fhe_v2
# Relinearization
~fhe_cdot

# vector with product, component to component
value_cdot = HE.decryptFrac(fhe_cdot)[:size]

# Error
expected_value = float(np.dot(v1, v2))
error = abs(expected_value - np.sum(value_cdot))

print(f"v1: {v1}")
print(f"v2: {v2}")
# cdot product is the sum for all the values in value_cdot
print(f"Homomorphic v1 * v2: {np.sum(value_cdot)}")
print(f"Expected value: {expected_value}")
print(f"Error: {error:.2e}")

v1: [ 90.79368301 -41.68964156 -41.74259603  90.10046964  97.00185146
 -50.17209384  84.62030586  16.88370296  -3.12720555  28.9128643 ]
v2: [ 56.30419847  47.03638124  32.50305504  71.77560614  62.18359004
 -46.88160277 -16.14551122 -98.7812279   70.84597392 -71.98579384]
Homomorphic v1 * v2: 11308.565549372968
Expected value: 11308.564718567051
Error: 8.31e-04


## How many operations can be done before data is corrupted?

We have seen that the multiplication operation adds noise to the data. For this reason is important to know how many operations can be done without corrupting data. This number  of operations is the operational limit which indicates the maximum number of operations that can be performed by the algorithm.

In CKKS the limit is the prime number chain defined in **qi_sizes = [60, 30, 30, 30, 30, 30, 60]**. With each rescaling operation one prime number is used, in this case we have five multiplication levels

In [9]:
from random import randrange

data = randrange(10)
fhe_data = HE.encryptFrac(np.array([data], dtype=np.float64))

print(f"Encrypting data = {data}")
print(f"Initial qi_sizes: {ckks_params['qi_sizes']}")
print(f"=> Available multiplicative depth: " f"{len(ckks_params['qi_sizes']) - 2}")

# Start multiplying encrypted data
for i in range(1, 10):
    try:
        fhe_data = fhe_data * fhe_data   
        # Relinearization
        ~fhe_data
        # Rescaling
        HE.rescale_to_next(fhe_data)
        expected_value = data ** (2 ** i)
        fhe_value = HE.decryptFrac(fhe_data)[0]
        error = abs(expected_value - fhe_value)
        print(f"Iteration {i}: data^{2**i:<5} | "
              f"Expected value={expected_value:<20.6f} "
              f"Encrypted value={fhe_value:<20.6f} | "
              f"Error={error:.2e}")
    except Exception as e:
        print(f"\nIteration {i}: FALLO - no more multiplication levels available.")
        print(f"  Excepción: {type(e).__name__}: {e}")
        break

Encrypting data = 9
Initial qi_sizes: [60, 30, 30, 30, 30, 30, 60]
=> Available multiplicative depth: 5
Iteration 1: data^2     | Expected value=81.000000            Encrypted value=80.999949            | Error=5.08e-05
Iteration 2: data^4     | Expected value=6561.000000          Encrypted value=6560.991763          | Error=8.24e-03
Iteration 3: data^8     | Expected value=43046721.000000      Encrypted value=43046612.918747      | Error=1.08e+02
Iteration 4: data^16    | Expected value=1853020188851841.000000 Encrypted value=1853010883774706.500000 | Error=9.31e+09
Iteration 5: data^32    | Expected value=3433683820292512441173561835520.000000 Encrypted value=-23177818728.277588  | Error=3.43e+30

Iteration 6: FALLO - no more multiplication levels available.
  Excepción: ValueError: scale out of bounds
